In [1]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
#options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_museums = []

driver.get("https://egymonuments.gov.eg/en/museums")

# Waiting for the museum links to load
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))
i = 1
while True:
    try:
        # press the next site link
        try:
            site_link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f'//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/a[{i}]')
            ))
            site_link.click()
            i += 1
        except:
            driver.find_element(By.XPATH, '//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/div/button').click()
            continue

        # collect the data
        place_name = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.mainPageTitle'))).text
        place_location = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.itemInfo'
        ).text
        place_description = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.txtSection'
        ).text

        try:
            place_photo = driver.find_element(By.CSS_SELECTOR,
                'body > div.innerMainBgCont > div > div:nth-child(1) > div > div.relative.descriptionImageCont > div.imageCont img'
            ).get_attribute('src')
        except:
            place_photo = ""

        try:
            start_from = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(1) > p'
            ).text
            end_at = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(2) > p'
            ).text
        except:
            start_from, end_at = "Not Available", "Not Available"

        try:
            tickets_price = driver.find_element(By.CSS_SELECTOR, 'div.ticketPriceDetails > div').text
        except:
            tickets_price = "Free"

        all_museums.append({
            "museum": place_name,
            "location": place_location,
            "Description": place_description,
            "start_from": start_from,
            "end_at": end_at,
            "tickets_price": tickets_price,
            "photo_url": place_photo,
            "on_map": ""
        })

        # return to the main page
        driver.back()
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))

    except Exception as e:
        print("Finished — no more items.")
        break

print(f"Total museums: {len(all_museums)}")

Finished — no more items.
Total museums: 24


In [5]:
df = pd.DataFrame(all_museums)
df

,museum,location,Description,start_from,end_at,tickets_price,photo_url,on_map
0,Marsa Matrouh Museum,Matrouh,This museum was established inside the Misr Pu...,08:00 AM,10:00 PM,FOREIGNER:\nAdult: EGP 80\ Student: EGP 40\n\n...,https://egymonuments.gov.eg/media/8793/picture...,
1,Graeco-Roman Museum,Alexandria,The Graeco-Roman Museum of Alexandria is one o...,09:00 AM,05:00 PM,FOREIGNER:\nAdult: EGP 400\ Student: EGP 200\n...,https://egymonuments.gov.eg/media/8211/whatsap...,
2,The National Museum of Suez,Suez,The idea for the National Museum of Suez was c...,09:00 AM,03:00 PM,FOREIGNERS:\nadults : EGP 180 \ students: EGP ...,https://egymonuments.gov.eg/media/7996/1203444...,
3,National Military Museum,Cairo,This museum is the first of its kind to be dev...,08:00 AM,04:00 PM,Area entry:\nFOREIGNERS:\nAdult: EGP 200 / Stu...,https://egymonuments.gov.eg/media/7904/g75a085...,
4,Helwan Corner Museum,Cairo,"In 1942, King Farouk built a rest house that c...",09:00 AM,05:00 PM,FOREIGNER:\nAdult: EGP 100 \ Student: EGP 50\n...,https://egymonuments.gov.eg/media/7665/7.jpg?a...,
5,Alexandria National Museum,Alexandria,The building that now houses the museum was or...,09:00 AM,05:00 PM,FOREIGNER:\nAdult: EGP 220\ Student: EGP 110\n...,https://egymonuments.gov.eg/media/7666/38916_1...,
6,Nubia Museum,Aswan,The Nubian Museum was founded in response to t...,09:00 AM,05:00 PM,FOREIGNER:\nAdult: EGP 400\ Student: EGP 200\n...,https://egymonuments.gov.eg/media/7664/60-1626...,
7,Crocodile Museum,Aswan,The Crocodile Museum is located next to the Ko...,07:00 AM,09:00 PM,Area entry:\nFOREIGNERS:\nAdult: EGP 450/ Stud...,https://egymonuments.gov.eg/media/7663/5728608...,
8,National Police Museum,Cairo,The National Police Museum is one of the museu...,08:00 AM,05:00 PM,Area entry:\nFOREIGNERS:\nAdult: EGP 200 / Stu...,https://egymonuments.gov.eg/media/7662/1.jpg?c...,
9,Cairo International Airport Museum - Terminal 2,Cairo,The idea of establishing a museum in Termina...,Not Available,Not Available,Opening Hours:\n24 hours daily.\n\nEgyptians: ...,https://egymonuments.gov.eg/media/7093/img-202...,


In [6]:
from selenium.webdriver.support import expected_conditions as EC

for idx, row in df.iterrows():
    query = f"{row['museum']}, {row['location']}, Egypt"
    print("Searching Maps for:", query)

    driver.get("https://www.google.com/maps/search/" + query.replace(" ", "+") + "?hl=en")

    try:
        WebDriverWait(driver, 15).until(
            lambda d: d.find_elements(By.CSS_SELECTOR, "h1.DUwDvf")
            or d.find_elements(By.CSS_SELECTOR, "div.Nv2PK")
        )
    except:
        pass

    if driver.find_elements(By.CSS_SELECTOR, "div.Nv2PK"):
        try:
            first_result = driver.find_element(By.CSS_SELECTOR, "div.Nv2PK a.hfpxzc")
            first_result.click()
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1.DUwDvf"))
            )
        except:
            pass

    time.sleep(1.5)

    df.at[idx, "on_map"] = driver.current_url

driver.quit()


Searching Maps for: Marsa Matrouh Museum, Matrouh, Egypt
Searching Maps for: Graeco-Roman Museum, Alexandria, Egypt
Searching Maps for: The National Museum of Suez, Suez, Egypt
Searching Maps for: National Military Museum, Cairo, Egypt
Searching Maps for: Helwan Corner Museum, Cairo, Egypt
Searching Maps for: Alexandria National Museum, Alexandria, Egypt
Searching Maps for: Nubia Museum, Aswan, Egypt
Searching Maps for: Crocodile Museum, Aswan, Egypt
Searching Maps for: National Police Museum, Cairo, Egypt
Searching Maps for: Cairo International Airport Museum - Terminal 2, Cairo, Egypt
Searching Maps for: The Egyptian Museum, Cairo, Egypt
Searching Maps for: Museum of Islamic Art, Cairo, Egypt
Searching Maps for: The Coptic Museum, Cairo, Egypt
Searching Maps for: National Museum of Egyptian Civilization(NMEC), Cairo, Egypt
Searching Maps for: Cairo International Airport Museum - Terminal 3, Cairo, Egypt
Searching Maps for: Sharm al-Sheikh Museum, South Sinai, Egypt
Searching Maps for

In [7]:
df.to_csv('museums_en.csv', index=False)